# all patients

In [1]:
# import features
import pandas as pd
# acitvity entropy
df_entropy = pd.read_csv('../output/data_cleaned/activity_entropy_rates.csv')
print(df_entropy.shape)

# early warning scores (physiological)
df_EWS = pd.read_csv('../output/data_cleaned/Mian_warning_score.csv')
df_EWS.columns = ['patient_id', 'date', 'early_warning_score']


# sleep quality
df_sleep_quality = pd.read_csv('../output/sleep_score/sleep_quality_score_by_duration.csv')
df_sleep_quality = df_sleep_quality.drop(columns= ['sum', 'mean','scaled_sleep_quality_sum']).copy()
df_sleep_quality.columns = ['patient_id', 'date', 'sleep_quality_score']

# agitation
df_agitation = pd.read_csv('../output/data_cleaned/agitation_daily_counts.csv')

# uti
df_uti = pd.read_csv('../output/data_cleaned/uti_daily.csv')

merged_df = df_entropy
# merge dataframes
for df in [df_EWS, df_sleep_quality, df_agitation, df_uti]:
    print(df.shape)
    merged_df = pd.merge(merged_df, df, on=['patient_id', 'date'], how='outer')
    
# only consider the patients without NA in the following analysis (as the analysis itself will be individualized anyway)
analysis_df = merged_df.dropna(subset=['sleep_quality_score']).dropna(subset=['early_warning_score']).dropna(subset=['entropy_rate']).copy()
analysis_df =  analysis_df.fillna(0)

print(analysis_df.shape)
analysis_df 

(2722, 3)
(2160, 3)
(800, 3)
(115, 3)
(265, 3)
(660, 7)


,patient_id,date,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen
219,0f352,2019-06-26,0.669008,0.0,2.233437,0.0,1.0
221,0f352,2019-06-28,0.613697,0.0,1.516700,0.0,0.0
222,0f352,2019-06-29,0.615494,0.0,-0.010261,0.0,1.0
223,0f352,2019-06-30,0.514768,0.0,2.607387,0.0,0.0
250,16f4b,2019-04-28,0.651879,0.0,-1.319084,0.0,0.0
...,...,...,...,...,...,...,...
2694,f220c,2019-06-06,0.620975,0.0,-2.627908,0.0,0.0
2696,f220c,2019-06-08,0.527123,0.0,2.607387,0.0,0.0
2706,f220c,2019-06-19,0.527442,0.0,2.607387,0.0,0.0
2709,f220c,2019-06-22,0.608841,0.0,2.607387,0.0,0.0


In [29]:
df_count_0 = pd.DataFrame(analysis_df.groupby("patient_id")["date"].count())
df_count = df_count_0.query('date >10').copy()


df_count

,date
patient_id,
16f4b,13
1fbe4,52
30a32,62
55cd4,64
93c14,36
96adf,49
a2849,57
c55f8,79
c5785,66


In [19]:
df_count["date"].median()

52.0

In [20]:
df_count["date"].max()

79

In [21]:
df_count["date"].min()

13

In [22]:
df_count["date"].quantile(0.75) - df_count["date"].quantile(0.25)

28.0

In [23]:
df_count["date"].sum()

633

# anomalies

In [24]:
import pandas as pd

ForestIsolation_sliding_01 = pd.read_csv('../output/Anomaly_delirium/forest_isolation_sliding_0.1contamination/forest_isolation_anomaly_data.csv')

df_LSTM_10sd = pd.read_csv('../output/Anomaly_delirium/LSTM_10days_baseline_1sd/LSTM_anomaly_data.csv')


print(ForestIsolation_sliding_01.shape)
print(df_LSTM_10sd.shape)

(71, 2)
(71, 8)


In [36]:
# forest isolation

df_count_fi = pd.DataFrame(ForestIsolation_sliding_01.groupby("patient_id")["date"].count())

df_count_fi = df_count_fi.rename(columns={
    "date": "date_anomaly"
})

df_count = df_count.rename(columns={"date": "date_total"})

df_all = pd.concat([df_count_fi, df_count], axis=1)
df_all["ratio_anomaly"] = df_all["date_anomaly"] / df_all["date_total"]
df_all                

,date_anomaly,date_total,ratio_anomaly
patient_id,,,
1fbe4,13.0,52,0.250000
30a32,14.0,62,0.225806
55cd4,9.0,64,0.140625
93c14,3.0,36,0.083333
96adf,5.0,49,0.102041
a2849,6.0,57,0.105263
c55f8,7.0,79,0.088608
c5785,8.0,66,0.121212
c8574,2.0,45,0.044444


In [37]:
df_all["ratio_anomaly"].median()



0.10204081632653061

In [25]:
df_LSTM_10sd

,patient_id,date,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen,anomaly
0,16f4b,2019-05-22,0.593706,0.0,1.625769,0.0,0.0,1.0
1,1fbe4,2019-05-06,0.668119,0.0,2.607387,0.0,0.0,1.0
2,1fbe4,2019-06-03,0.708331,1.0,-1.319084,0.0,1.0,1.0
3,1fbe4,2019-06-13,0.564695,2.0,-0.010261,0.0,1.0,1.0
4,1fbe4,2019-06-21,0.700437,2.0,-1.755359,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...
66,ec812,2019-06-06,0.698471,0.0,1.655515,0.0,0.0,1.0
67,ec812,2019-06-19,0.761631,0.0,0.316945,0.0,0.0,1.0
68,ec812,2019-06-22,0.623560,0.0,0.862288,0.0,0.0,1.0
69,f220c,2019-06-08,0.527123,0.0,2.607387,0.0,0.0,1.0
